In [24]:
from docx import Document
from docx.shared import Inches
from docx.enum.text import WD_PARAGRAPH_ALIGNMENT
from docx.oxml.ns import qn
from docx.oxml import OxmlElement
import os
import pandas as pd
import ast
import platform

In [4]:
df_brca = pd.read_csv('brca_data.csv')
df_brca.head()

,label,symptoms,Patient_ID,file_path,images
0,Benign,Multiple retroareolar rounded equal density ma...,311L,Medical_reports_for_cases/P311.docx,['/home/mcetin/mm_rag_brca/chatbot/resized_sub...
1,Malignant,Upper outer quadrant irregular high density ma...,311R,Medical_reports_for_cases/P311.docx,['/home/mcetin/mm_rag_brca/chatbot/resized_sub...
2,Malignant,Diffuse edematous changes evidenced by increas...,28L,Medical_reports_for_cases/P28.docx,['/home/mcetin/mm_rag_brca/chatbot/resized_sub...
3,Benign,Upper outer quadrant asymmetrical increased de...,28R,Medical_reports_for_cases/P28.docx,['/home/mcetin/mm_rag_brca/chatbot/resized_sub...
4,Malignant,Upper outer asymmetrical increased density ass...,223L,Medical_reports_for_cases/P223.docx,['/home/mcetin/mm_rag_brca/chatbot/resized_sub...


In [28]:
def create_img_anotation(doc, patient_id, annotation, image1=None, image2=None):
    # Create a new Document
    # doc = Document()
    doc.add_heading('Patient Report: ' + patient_id, level=1)

    # Add a title for the images
    doc.add_heading('Mammogram Images', level=2)

    # Add a table to hold the images side by side
    table = doc.add_table(rows=2, cols=2)
    table.autofit = True

    # Add the first image to the first cell
    cell1 = table.cell(0, 0)
    paragraph1 = cell1.paragraphs[0]
    run1 = paragraph1.add_run()
    run1.add_picture(image1, width=Inches(2.5))

    # Add title below the first image
    cell1_title = table.cell(1, 0)
    cell1_title.text = image1.split("_")[-1].split(".")[0]

    if image2 is not None:
        # Add the second image to the second cell
        cell2 = table.cell(0, 1)
        paragraph2 = cell2.paragraphs[0]
        run2 = paragraph2.add_run()
        run2.add_picture(image2, width=Inches(2.5))

        # Add title below the second image
        cell2_title = table.cell(1, 1)
        cell2_title.text = image2.split("_")[-1].split(".")[0]

    # Add a paragraph of notes about the pictures
    doc.add_paragraph(annotation)
    
    return doc

def create_word_document(df_brca, file_path):
    doc = Document()

    for indx in df_brca.index[:5]:
        df_brca.loc[indx]

        patient_id = df_brca.loc[indx, 'Patient_ID']
        symptoms = df_brca.loc[indx, 'symptoms']

        # Convert the string representation to an actual list for a specific cell
        images_list = ast.literal_eval(df_brca.loc[indx, "images"])

        image_1 = images_list[0]
        image_2 = None
        if len(images_list) > 1:
            image_2 = images_list[1]
        
        print("Patient_id: ", patient_id)
        doc = create_img_anotation(doc, patient_id, symptoms, image1=image_1, image2=image_2)
        
        # Add a page break
        doc.add_page_break()


    # # Create the file path
    # file_path = os.path.join(output_folder, "all.docx")
        
    # Save the document
    doc.save(file_path)
    print("Document saved at: ", file_path)

In [53]:
def create_pdf_from_docx(file_path):
    output_folder, file_name = os.path.split(file_path)
    pdf_path = file_path.replace(".docx", ".pdf")
    # Check if the operating system is Windows and convert to PDF if true
    if platform.system() == "Windows":
        from docx2pdf import convert
        convert(file_path, pdf_path)
        print(f"Converted {file_path} to {pdf_path}")
        
    elif platform.system() == "Linux":
        # os.system(f'cd {output_folder} && libreoffice --headless --convert-to pdf {file_name}')
        _ = os.system(f'cd {output_folder} && libreoffice --headless --convert-to pdf {file_name} > /dev/null 2>&1')
        print(f"Converted {file_path} to {pdf_path}")
        
    else:
        print("Unsupported operating system.")

In [ ]:
output_folder = 'create_image_and_annotation_folder'
file_name = 'patient_all.docx'

# Create the file path
file_path = os.path.join(output_folder, file_name)
    
create_word_document(df_brca, file_path)

# create_pdf_from_docx(file_path)

Patient_id:  311L
Patient_id:  311R
Patient_id:  28L
Patient_id:  28R
Patient_id:  223L
Document saved at:  create_image_and_annotation_folder/patient_all.docx


In [59]:
create_pdf_from_docx(file_path)

Converted create_image_and_annotation_folder/patient_all.docx to create_image_and_annotation_folder/patient_all.pdf
